Welcome to the first big project done on my new local workhorse computer!<br>
# Our Problem Statement
Natural language processing is an important part of the most advanced artificial intelligence software we have today. By studying volumes of text, word embeddings are able to elicit meaning from the words within training data. Your goal is to train a word embedding on three famous works of Shakespeare to determine how well your embedding can understand the meaning of character names and other Shakespearean English words found in these plays.

## Step 1.1, Gathering our data

In [1]:
# importing language library and database of our data
import nltk 

In [2]:
nltk.download('gutenberg')# as this is a new environment we need to download the database
# will say it is simply up to date in later iterations

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Danton\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


True

In [3]:
# ensuring I have all other necessary libraries for this process
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
# if I have them they will simply return they are up to date

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Danton\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Danton\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Danton\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Danton\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
# with everything up to date and ensuring I have everything I need to work from this point on
# lets begin!
from nltk.corpus import gutenberg # our lib
from autocorrect import Speller # for checking spelling
from nltk.stem import WordNetLemmatizer # for transforming words to their base form
# changing 'changing' to 'change'. We are changing words to their base dictionary forms

from nltk.corpus import stopwords # for removing stopwords, non important words to context, 'the, as, etc' 
# these do not add much context to the data which can slow down machines when processing, better to remove them for most cases

import re
from nltk.tokenize import sent_tokenize, word_tokenize # to transform our text into machine readable formats


In [5]:
#loading our data into a single list variable to combine later
plays = [
    gutenberg.raw('shakespeare-hamlet.txt'), # Hamlet
    gutenberg.raw('shakespeare-macbeth.txt'), # MacBeth
    gutenberg.raw('shakespeare-caesar.txt')  # Julius Caesar
]

In [6]:
# a quick test with data I am familiar with to ensure accuracy for later
plays[2][:300]
# wow this is a very old version!

'[The Tragedie of Julius Caesar by William Shakespeare 1599]\n\n\nActus Primus. Scoena Prima.\n\nEnter Flauius, Murellus, and certaine Commoners ouer the Stage.\n\n  Flauius. Hence: home you idle Creatures, get you home:\nIs this a Holiday? What, know you not\n(Being Mechanicall) you ought not walke\nVpon a la'

In [7]:
# so we have lots of newlines and many different odd spellings, thanks shakespeare!

#combining the plays into a single variable
data = ' '.join(plays).lower()  # Lowercase the text and joining the plays together into a single string

In [8]:
# Now we tokenize the data into sentences
sentences = sent_tokenize(data)# this splits the data into a list of sentences

# Then we reuse our data variable and place the new worlist 
data = [word_tokenize(sentence) for sentence in sentences]

Why not convert directly to words? Catching sentences first allows us to capture context 
that might have been missed by converting everything to words. Puncuation and special characters matter in this case and converting directly to words may cause errors, especially in noisy data. Which anything from Shakespeare should automatically be considered as noisy!<br>
Otherwise this can be a bit more memory efficient, giving our system a break. 

In [9]:
# initializing our spellcheck system
speller = Speller(lang='en')

In [10]:
# now we spell check, to ensure any words aren't mispelled for the lemmatizer
data_spell = [
    [speller(word) for word in sentence]
    for sentence in data
]

In [11]:
# Removing any remaining non-alphabetical tokens using our re lib
data_re = [
    [re.sub(r'[^a-zA-Z]', '', word) for word in sentence if re.sub(r'[^a-zA-Z]', '', word)]
    for sentence in data_spell
]

In [12]:
# initializing our Stopwords list
stop_words = set(stopwords.words('english'))

In [13]:
# now we are removing stopwords that are deemed not important!
data_word = [
    [word for word in sentence if word not in stop_words]
    for sentence in data_re
]

In [14]:
# initializing our lemmatizer
lemmatizer = WordNetLemmatizer()

In [15]:
# now we use our lemmetizer to change words to their base dictionary forms
data_processed = [
    [lemmatizer.lemmatize(word) for word in sentence]
    for sentence in data_word
]

In [16]:
# checking our first 5 sentences
print(data_processed[:5])

[['tragedy', 'hamlet', 'william', 'shakespeare', 'act', 'prime'], ['scene', 'prima'], ['enter', 'bernard', 'francisco', 'two', 'sentinel'], ['bernard'], []]


In [17]:
plays[0][:200]# for checking accuracy of above statement

"[The Tragedie of Hamlet by William Shakespeare 1599]\n\n\nActus Primus. Scoena Prima.\n\nEnter Barnardo and Francisco two Centinels.\n\n  Barnardo. Who's there?\n  Fran. Nay answer me: Stand & vnfold\nyour sel"

I had to reorder a few steps as potentially there were some spellings that made it more difficult for 
our lemmatizer to convert. Otherwise I am attempting to remove some excess noise that is present after cleaning.

## Step 2.1, Modelling

### Word2Vec model

In [18]:
#importing our model
from gensim.models import word2vec

In [19]:
# initializing our model
# cbow stands for Continuous Bag of Words
cbow = word2vec.Word2Vec(
    sentences=data_processed,
    vector_size=200,  # what size the output is going to be
    window=10, # how big of a window(range) of words we are looking at for context
    min_count=3, # how many times a word must appear before we are to consider it valuable
    sg=0, #specifying that this is for a cbow
    epochs=20
)

CBOW is a method of word vectorization in Word2Vec where the model learns to predict a target word based on its surrounding context words. It does this through a neural network that outputs a probability distribution over vocabulary words, with the output layer typically utilizing a softmax function. Softmax transforms the model's output vector into a probability distribution across all possible words, helping identify the most likely target word given the context. The resulting vector for each word encodes its relationships with other words based on co-occurrence patterns, allowing the comparison of word vectors to determine contextual similarity.

In [20]:
# training the model
cbow.train(data_processed, total_examples=len(data_processed), epochs=20)

(549445, 700460)

In [21]:
# lets see how well this worked by gathering the top 20 words from the model
top_20_words = list(cbow.wv.key_to_index.keys())[:20]
top_20_counts = [cbow.wv.get_vecattr(word, 'count') for word in top_20_words]

In [22]:
# Print results
for word, count in zip(top_20_words, top_20_counts):
    print(f"{word}: {count}")

ham: 337
thou: 307
lord: 306
shall: 300
come: 284
king: 248
enter: 230
good: 221
let: 220
mac: 205
thy: 202
like: 200
cesar: 193
one: 188
make: 185
know: 184
v: 184
thee: 174
self: 166
would: 163


What is the v for? Is it another remnant from the cleaning? I thought those were removed. Possibly I 
need to set up a system to clean up single variable words as they seem to be left from other data cleanings that were done. Unless it is a name of someone. Lets take a look!

In [23]:
sentence_list = []
orgin = []
index=0
for sentence in data_processed:
    for word in sentence:
        # Check if the word length is 1
        if len(word) == 1:
            orgin.append(index)
            sentence_list.append(sentence)
    index=index+1
print(sentence_list[0:5])
# seeing the output v is coming from macbeth. Interesting. Lets look to see if we can't find the specific entries

[['ratio', 'sale', 'fantasy', 'let', 'believe', 'take', 'hold', 'touching', 'dreaded', 'sight', 'twice', 'seen', 'v', 'therefore', 'untreated', 'along', 'v', 'watch', 'minute', 'night', 'apparition', 'come', 'may', 'approve', 'eye', 'speak', 'hor'], ['ratio', 'sale', 'fantasy', 'let', 'believe', 'take', 'hold', 'touching', 'dreaded', 'sight', 'twice', 'seen', 'v', 'therefore', 'untreated', 'along', 'v', 'watch', 'minute', 'night', 'apparition', 'come', 'may', 'approve', 'eye', 'speak', 'hor'], ['sit', 'awhile', 'let', 'v', 'assault', 'ear', 'fortified', 'story', 'two', 'night', 'seen', 'hor'], ['well', 'sit', 'let', 'v', 'bernard', 'speak', 'barn'], ['least', 'whisper', 'go', 'last', 'king', 'whose', 'image', 'even', 'appear', 'v', 'know', 'fortinbras', 'norway', 'thereto', 'price', 'emulate', 'pride', 'dar', 'combat']]


In [24]:
print(orgin[0:5])

[39, 39, 41, 42, 73]


In [25]:
print(sentences[39])
print(sentences[73])

horatio saies, 'tis but our fantasie,
and will not let beleefe take hold of him
touching this dreaded sight, twice seene of vs,
therefore i haue intreated him along
with vs, to watch the minutes of this night,
that if againe this apparition come,
he may approue our eyes, and speake to it

   hor.
that can i,
at least the whisper goes so: our last king,
whose image euen but now appear'd to vs,
was (as you know) by fortinbras of norway,
(thereto prick'd on by a most emulate pride)
dar'd to the combate.


Hmm. Seems that v is simply a version 'us' as shakespeare is the worst when it comes to text cleaning 
since he liked to mess with spelling and pronounciation all the time. He invented over 1700 unique words unfortunately. 

In [26]:
# Initialize the skip-gram Word2Vec model
skipgram = word2vec.Word2Vec(
    sentences=data_processed,          # our text data
    vector_size=300,                   # dimensionality of the vectors
    window=5,                          # context window size, trying a smaller window
    min_count=3,                       # ignores words with lower frequency than 5
    sg=1                               # 1 for skip-gram; 0 for CBOW, this affects what approach/algorithm is chosen
)

The Skip-gram model in Word2Vec learns to predict surrounding context words given a target word. Unlike CBOW, which predicts a target word based on its surrounding words, Skip-gram takes a single word as input and tries to predict the words likely to appear around it. This approach allows Skip-gram to capture a word’s relationship to its context more effectively in sparse data settings and results in word vectors that encode semantic similarities based on co-occurrences in various contexts.

In [27]:
# Train the model for 20 number of epochs
skipgram.train(data_processed, total_examples=len(data_processed), epochs=20)

(549833, 700460)

In [28]:
# now loading in a pretrained model as required in our instructions
from gensim.models import KeyedVectors
GloVe = KeyedVectors.load_word2vec_format("glove.6B.200d.txt", no_header=True)
# using the 200 vector dimension version cause I can

I am using the 6B dataset, which is a model trained on the entirety of Wikipedia 2014 and Gigaword 5. 
Gigaword 5 is a collection of text taken from news articles between January 2009 and December 2010. 

## Step 3.1, Comparisons
Compare the three models by finding the 5 most similar terms to each of the following terms: 'hamlet', 'cauldron', 'nature', 'spirit', 'general', and 'prythee'.

In [29]:
#Gathering word vectors similar to our given word
similar_words_cbow = cbow.wv.most_similar('hamlet', topn=5)
similar_words_skip = skipgram.wv.most_similar('hamlet', topn=5)
similar_words_glove = GloVe.most_similar('hamlet', topn=5)

index = 0
models = ['cbow','skipgram','GloVe']
for words in [similar_words_cbow,similar_words_skip,similar_words_glove]:
    print(f'Model {models[index]} comparing to Hamlet/n', words, '\n')
    index= index+1

Model cbow comparing to Hamlet/n [('alert', 0.9211762547492981), ('sweet', 0.8814935684204102), ('queen', 0.8580957651138306), ('denmark', 0.8572719693183899), ('cousin', 0.8554648756980896)] 

Model skipgram comparing to Hamlet/n [('alert', 0.8635126352310181), ('cousin', 0.8182710409164429), ('queen', 0.8103752732276917), ('sent', 0.8077291250228882), ('ophelia', 0.802406907081604)] 

Model GloVe comparing to Hamlet/n [('village', 0.595661461353302), ('town', 0.5756247639656067), ('situated', 0.5134664177894592), ('hamlets', 0.4892188310623169), ('towns', 0.48087748885154724)] 



Facinating. We see our trained model is lost for Shakespeare as it is using definition of a hamlet 
instead of referring to a character, where our models are referring to the character Hamlet. We see he was quite alert as both models found Alert to be very similar in vector angle. If I am not mistaken, Ophelia is his wife (Potential Wife) which makes sense as their stories are closely tied together. Rosincrane, which is not the characters name so some text cleaning went very wrong, is a character whom acts as a courier/diplomat in Hamlet to get closer to the king of the time which makes sense why they are related. Seems Skip Gram was able to capture the relations between characters quite well. 

In [30]:
#Gathering word vectors similar to our given word
similar_words_cbow = cbow.wv.most_similar('cauldron', topn=5)
similar_words_skip = skipgram.wv.most_similar('cauldron', topn=5)
similar_words_glove = GloVe.most_similar('cauldron', topn=5)

index = 0
models = ['cbow','skipgram','GloVe']
for words in [similar_words_cbow,similar_words_skip,similar_words_glove]:
    print(f'Model {models[index]} comparing to Cauldron/n', words, '\n')
    index= index+1

Model cbow comparing to Cauldron/n [('toile', 0.9913268685340881), ('bubble', 0.9748231172561646), ('burned', 0.9608901739120483), ('wolf', 0.9444016218185425), ('dropping', 0.9396466016769409)] 

Model skipgram comparing to Cauldron/n [('bubble', 0.9712512493133545), ('toile', 0.9605293273925781), ('drive', 0.913554310798645), ('burned', 0.8883442282676697), ('shadow', 0.8867537379264832)] 

Model GloVe comparing to Cauldron/n [('caldron', 0.757416844367981), ('flame', 0.5621885657310486), ('cauldrons', 0.5294731855392456), ('seething', 0.48947516083717346), ('torch', 0.480618953704834)] 



If I remember, the caudron in McBeth was quite the important object in act.... 3? I cannot remember but potentially the caudron had an encantation said while within the scene which included toile and bubble, possibly trouble. Looked it up, Likely was originally Bubble Bubble Toile and Trouble! Cool! CBOW takes this one for the context of the plays as it registers the witches words quite well, but GloVe still gives us words more associated with cauldrons.

In [31]:
#Gathering word vectors similar to our given word
similar_words_cbow = cbow.wv.most_similar('nature', topn=5)
similar_words_skip = skipgram.wv.most_similar('nature', topn=5)
similar_words_glove = GloVe.most_similar('nature', topn=5)

index = 0
models = ['cbow','skipgram','GloVe']
for words in [similar_words_cbow,similar_words_skip,similar_words_glove]:
    print(f'Model {models[index]} comparing to Nature/n', words, '\n')
    index= index+1

Model cbow comparing to Nature/n [('seems', 0.8853791356086731), ('life', 0.8790339231491089), ('whose', 0.8735732436180115), ('mortal', 0.8723843097686768), ('head', 0.8704405426979065)] 

Model skipgram comparing to Nature/n [('mortal', 0.7911329865455627), ('seems', 0.782158613204956), ('abuse', 0.7693685293197632), ('custom', 0.7643863558769226), ('action', 0.762302041053772)] 

Model GloVe comparing to Nature/n [('aspects', 0.6314504146575928), ('natural', 0.61203932762146), ('particular', 0.6074705719947815), ('subject', 0.6006618142127991), ('rather', 0.5964077711105347)] 



Our trained models are both again looking at nature within the Shakespearian context. SkipGram has the 
better results here but GloVe still has a much stronger relation to it, giving us the idea that nature is to describe ones personality or attributes perhaps. Interesting. 

In [32]:
#Gathering word vectors similar to our given word
similar_words_cbow = cbow.wv.most_similar('spirit', topn=5)
similar_words_skip = skipgram.wv.most_similar('spirit', topn=5)
similar_words_glove = GloVe.most_similar('spirit', topn=5)

index = 0
models = ['cbow','skipgram','GloVe']
for words in [similar_words_cbow,similar_words_skip,similar_words_glove]:
    print(f'Model {models[index]} comparing to Spirit/n', words, '\n')
    index= index+1

Model cbow comparing to Spirit/n [('fall', 0.8816090226173401), ('blood', 0.8615347743034363), ('cursed', 0.8569236993789673), ('point', 0.8206045031547546), ('concur', 0.8163424730300903)] 

Model skipgram comparing to Spirit/n [('bone', 0.7273350954055786), ('settle', 0.7253388166427612), ('wholesome', 0.7201343178749084), ('deeper', 0.7130470275878906), ('draw', 0.7129603028297424)] 

Model GloVe comparing to Spirit/n [('faith', 0.5849964022636414), ('belief', 0.5739845633506775), ('embodied', 0.5735374093055725), ('spirits', 0.57210373878479), ('passion', 0.5700939893722534)] 



CBOw takes the win in our home trained models, as it is showing more spiritually linked terms, as 
many of the terms given can be used in very spiritual contexts, especially by Shakespeare!<br>
GloVe gives us a plural form and interestingly religious terms for Spirit, as Faith and belief are an understandable link but a mildly surprising one as I figured determination or a varient of that would be present. 

In [33]:
#Gathering word vectors similar to our given word
similar_words_cbow = cbow.wv.most_similar('general', topn=5)
similar_words_skip = skipgram.wv.most_similar('general', topn=5)
similar_words_glove = GloVe.most_similar('general', topn=5)

index = 0
models = ['cbow','skipgram','GloVe']
for words in [similar_words_cbow,similar_words_skip,similar_words_glove]:
    print(f'Model {models[index]} comparing to General/n', words, '\n')
    index= index+1

Model cbow comparing to General/n [('passe', 0.9541304111480713), ('drop', 0.9490125775337219), ('vile', 0.9471206665039062), ('stirred', 0.9366024732589722), ('pope', 0.9355875253677368)] 

Model skipgram comparing to General/n [('yee', 0.8945964574813843), ('therein', 0.8762234449386597), ('beat', 0.8564705848693848), ('count', 0.8557938933372498), ('constant', 0.8505608439445496)] 

Model GloVe comparing to General/n [('gen.', 0.6491652131080627), ('secretary', 0.6104984879493713), ('chief', 0.5887365341186523), ('command', 0.5681266784667969), ('deputy', 0.5612561106681824)] 



First off, GloVe gives us related roles and an abbreviation of General. Neat. <br>
CBOW is better at this one, as it is capturing more relations with General but misses the shakespearian twist Therein, as therein is a general 'all that' term. Interesting relation there but more literal relations are captured via CBOW. 

In [34]:
#Gathering word vectors similar to our given word
similar_words_cbow = cbow.wv.most_similar('prythee', topn=5)
similar_words_skip = skipgram.wv.most_similar('prythee', topn=5)
try:
    similar_words_glove = GloVe.most_similar('prythee', topn=5)
except:
    similar_words_glove = 'No words are within this dictionary'
index = 0
models = ['cbow','skipgram','GloVe']
for words in [similar_words_cbow,similar_words_skip,similar_words_glove]:
    print(f'Model {models[index]} comparing to Prythee/n', words, '\n')
    index= index+1

Model cbow comparing to Prythee/n [('fellow', 0.9415457248687744), ('conspiracy', 0.9213850498199463), ('stratum', 0.9167755842208862), ('farewell', 0.9025022387504578), ('nod', 0.9008446335792542)] 

Model skipgram comparing to Prythee/n [('character', 0.9263906478881836), ('saying', 0.9200066328048706), ('nod', 0.9063267111778259), ('keen', 0.9053429365158081), ('conspiracy', 0.9030908346176147)] 

Model GloVe comparing to Prythee/n No words are within this dictionary 



Oh boy! GloVe didn't have anything on Shakespeare! I think Prythee may be better captured in CBOW as 
the context of what is said via Prythee is represented within the most related words given. 

## Step 3.2, Word to Word Comparisons

In [35]:
# defining our pairs for the next stage of comparisions
term_pairs = [
    ('brutus', 'murder'),
    ('lady macbeth', 'queen gertrude'),
    ('fortinbras', 'norway'),
    ('rome', 'norway'),
    ('ghost', 'spirit'),
    ('macbeth', 'hamlet')
]

In [36]:
# Getting Word vector from a model
def get_embedding(model, word):
    try:
        return model[word]# getting vector from GloVe
    except TypeError:
        try:
            return model.wv[word] # getting vector from Not GloVe
        except KeyError:
            return None # gives us an null value if word is not found
    except KeyError:
        return None

In [37]:
# looping through our pairs to set up our cosine similarities for CBOW
# Could not get a system to work for all three models dynamically sadly. 
for i in range(0, len(term_pairs)):
    v1 = get_embedding(cbow, term_pairs[i][0])
    v2 = get_embedding(cbow, term_pairs[i][1])
    
    # Compute similarities model
    #copied from textbook
    if v1 is not None and v2 is not None:
        cbow_sim = cbow.wv.cosine_similarities(v1, [v2])
    else:
       cbow_sim  = "Words not found withn Model Corpus."    

    print(f'The CBOW Models cosine similarity between {term_pairs[i]} is: ', cbow_sim)

The CBOW Models cosine similarity between ('brutus', 'murder') is:  Words not found withn Model Corpus.
The CBOW Models cosine similarity between ('lady macbeth', 'queen gertrude') is:  Words not found withn Model Corpus.
The CBOW Models cosine similarity between ('fortinbras', 'norway') is:  [0.956262]
The CBOW Models cosine similarity between ('rome', 'norway') is:  [0.09452884]
The CBOW Models cosine similarity between ('ghost', 'spirit') is:  [-0.06130803]
The CBOW Models cosine similarity between ('macbeth', 'hamlet') is:  [0.20882666]


In [38]:
# Could not get a system to work for all three models dynamically sadly. 
# Reusing this loop and all variables but changing for SkipGram
for i in range(0, len(term_pairs)):
    v1 = get_embedding(skipgram, term_pairs[i][0])# gathering word vectors
    v2 = get_embedding(skipgram, term_pairs[i][1])
    
    # Compute similarities model
    #copied from textbook
    if v1 is not None and v2 is not None:
        cbow_sim = skipgram.wv.cosine_similarities(v1, [v2])
    else:
       cbow_sim  = "Words not found withn Model Corpus."
    
    print(f'The SkipGram Models cosine similarity between {term_pairs[i]} is: ', cbow_sim)

The SkipGram Models cosine similarity between ('brutus', 'murder') is:  Words not found withn Model Corpus.
The SkipGram Models cosine similarity between ('lady macbeth', 'queen gertrude') is:  Words not found withn Model Corpus.
The SkipGram Models cosine similarity between ('fortinbras', 'norway') is:  [0.90253794]
The SkipGram Models cosine similarity between ('rome', 'norway') is:  [0.320206]
The SkipGram Models cosine similarity between ('ghost', 'spirit') is:  [0.24408205]
The SkipGram Models cosine similarity between ('macbeth', 'hamlet') is:  [0.26556715]


In [39]:
#Could not get a system to work for all three models dynamically sadly. 
# Reusing this loop and all variables but changing for GloVe
for i in range(0, len(term_pairs)):
    v1 = get_embedding(GloVe, term_pairs[i][0])
    v2 = get_embedding(GloVe, term_pairs[i][1])
    
    # Compute similarities model
    #copied from textbook
    if v1 is not None and v2 is not None:
        cbow_sim = GloVe.cosine_similarities(v1, [v2])
    else:
       cbow_sim  = "Words not found withn Model Corpus."
    
    print(f'The GloVe Models cosine similarity between {term_pairs[i]} is: ', cbow_sim)

The GloVe Models cosine similarity between ('brutus', 'murder') is:  [0.10258823]
The GloVe Models cosine similarity between ('lady macbeth', 'queen gertrude') is:  Words not found withn Model Corpus.
The GloVe Models cosine similarity between ('fortinbras', 'norway') is:  [0.04613631]
The GloVe Models cosine similarity between ('rome', 'norway') is:  [0.13575909]
The GloVe Models cosine similarity between ('ghost', 'spirit') is:  [0.35051912]
The GloVe Models cosine similarity between ('macbeth', 'hamlet') is:  [0.3907806]


It is odd that both our home grown models did not have brutus and murder within their corpuses. 
We see a common trend that multi-term words are not accepted by all models but we would likely get results for McBeth and Gertrude individually. Otherwise we see quite a few minor relations (as values towards 0 indicate lower relations or no relations). Funny enough Fortinbras and Norway is very good there is a strong relation as Fortinbras is Norweigen in Hamlet.  Rome and Norway likely don't have enough context for our models to set strong relations but even GloVe does not have a strong relation for them so our models are doing fine. Ghost and Spirit's relation are well captured by our models but GloVe also shows the strongest relation to McBeth and Hamlet, suggesting there is likely more underlying context between the two in the data GloVe was trained on. 

## Step 3.3, Word Vector Comparisons and Modifications
Compare the three models by finding the 5 most similar terms to each of the following word vectors obtained via linear combination: <br>'denmark' + 'queen'<br>, 'scotland' + 'army' + 'general'<br>, 'father' - 'man' + 'woman'<br>, 'mother' - 'woman' + 'man'<br>. Comment on how well each model described the ideas behind these word vectors.

In [40]:
# This works by adding or subtracting the word vectors essentially. Really cool to see relations
print('Denmark and Queen:\n', cbow.wv.most_similar(positive=['denmark', 'queen'], topn=5), '\n')
print('Scotland, Army, and General:\n', cbow.wv.most_similar(positive=['scotland', 'army', 'general'], topn=5), '\n')
print('Father minus Man and Woman:\n', cbow.wv.most_similar(positive=['father'], negative=['man', 'woman'], topn=5), '\n')
print('Mother minus Man and Woman:\n', cbow.wv.most_similar(positive=['mother'], negative=['man', 'woman'], topn=5), '\n')

Denmark and Queen:
 [('ophelia', 0.9096110463142395), ('alert', 0.9062366485595703), ('hamlet', 0.9032272696495056), ('qu', 0.8988021612167358), ('poison', 0.8959996700286865)] 

Scotland, Army, and General:
 [('flourish', 0.9791468977928162), ('diver', 0.9784242510795593), ('hack', 0.9773777723312378), ('trust', 0.9768217206001282), ('sacred', 0.9752756953239441)] 

Father minus Man and Woman:
 [('hamlet', 0.44066861271858215), ('alert', 0.4262400269508362), ('ratio', 0.3481651842594147), ('queen', 0.3151561915874481), ('gertrude', 0.302213579416275)] 

Mother minus Man and Woman:
 [('alert', 0.47997310757637024), ('hamlet', 0.46273577213287354), ('gertrude', 0.44415730237960815), ('ratio', 0.43842530250549316), ('either', 0.3636581003665924)] 



In [41]:
# This works by adding or subtracting the word vectors essentially. Really cool to see relations
print('Denmark and Queen:\n', skipgram.wv.most_similar(positive=['denmark', 'queen'], topn=5), '\n')
print('Scotland, Army, and General:\n', skipgram.wv.most_similar(positive=['scotland', 'army', 'general'], topn=5), '\n')
print('Father minus Man and Woman:\n', skipgram.wv.most_similar(positive=['father'], negative=['man', 'woman'], topn=5), '\n')
print('Mother minus Man and Woman:\n', skipgram.wv.most_similar(positive=['mother'], negative=['man', 'woman'], topn=5), '\n')

Denmark and Queen:
 [('poison', 0.8988074064254761), ('alert', 0.8900041580200195), ('rosincrane', 0.8863423466682434), ('killed', 0.8818949460983276), ('cousin', 0.881440281867981)] 

Scotland, Army, and General:
 [('condemn', 0.9316120147705078), ('finance', 0.9278732538223267), ('marching', 0.9275248646736145), ('flourish', 0.926675021648407), ('recorder', 0.9264633655548096)] 

Father minus Man and Woman:
 [('heaven', 0.13653747737407684), ('look', 0.13098204135894775), ('queen', 0.12087567895650864), ('tonne', 0.11848986148834229), ('lost', 0.11725771427154541)] 

Mother minus Man and Woman:
 [('gertrude', 0.19596408307552338), ('father', 0.18269748985767365), ('queen', 0.1766403317451477), ('qu', 0.15853212773799896), ('heaven', 0.13213208317756653)] 



In [42]:
# This works by adding or subtracting the word vectors essentially. Really cool to see relations
print('Denmark and Queen:\n', GloVe.most_similar(positive=['denmark', 'queen'], topn=5), '\n')
print('Scotland, Army, and General:\n', GloVe.most_similar(positive=['scotland', 'army', 'general'], topn=5), '\n')
print('Father minus Man and Woman:\n', GloVe.most_similar(positive=['father'], negative=['man', 'woman'], topn=5), '\n')
print('Mother minus Man and Woman:\n', GloVe.most_similar(positive=['mother'], negative=['man', 'woman'], topn=5), '\n')

Denmark and Queen:
 [('sweden', 0.6906564831733704), ('king', 0.6258984804153442), ('norway', 0.6186422109603882), ('princess', 0.596460223197937), ('danish', 0.5936761498451233)] 

Scotland, Army, and General:
 [('military', 0.6814087629318237), ('forces', 0.6494377851486206), ('command', 0.6411989331245422), ('commander', 0.6330193877220154), ('lieutenant', 0.6149927377700806)] 

Father minus Man and Woman:
 [('orbón', 0.5008911490440369), ('daynes', 0.5005658268928528), ('weißheimer', 0.46633175015449524), ('roquelaure', 0.4611763060092926), ('purnananda', 0.457046777009964)] 

Mother minus Man and Woman:
 [('daynes', 0.4715067148208618), ('kumler', 0.4624353051185608), ('boskey', 0.4556521475315094), ('pellerano', 0.4527418911457062), ('orbón', 0.44916173815727234)] 



Wow! We know the Queen of Denmark, Hamlets mother, drinks poison and is murdered in the place of her 
son. Seeing that our models have relations for these words is interesting as it shows a basic comprehension of sentence context over a word by word basis. GloVe has far more broad terms linked to eachother but it is also much larger in scale and training when compared to our models. We see some odd terms related to eachother but they make sense as both our models are working within the context of the three Shakespeare tragedies, so some relations may not make as much sense to us without larger context. 

## Step 3.4, Final Conclusions
Well, both our homegrown models did well within their context but are very limited as such. GloVe shows there are many different relations for words that we may not think of initially, as seen above for parents (We never see parent as a term oddly). Which model preforms best is hard to say, I think CBOW was better for most cases but some tuning of hyperparameters and SkipGram will likely surpase it as SkipGram is better at capturing larger sentence contexts. We see that thoughout the notebook as SkipGram had better word relations often but CBOW had better  relations over multiple words. We do have different windows for the models and that is a factor in their proformance